In [5]:
import pandas as pd
import numpy as np
import time as time_lib
import warnings
warnings.filterwarnings("ignore")

In [6]:
import sklearn.neighbors._base
import sys
sys.modules['sklearn.neighbors.base'] = sklearn.neighbors._base

In [12]:
from missingpy import MissForest
imputer = MissForest()

In [13]:
n_items_list = [50, 100, 100, 100, 500, 500, 1000, 1000, 1000, 1500, 1500]
m_users_list = [50, 100, 300, 5+00, 100, 500, 100, 200, 300, 1000, 2000]

for n_items, m_users in zip(n_items_list, m_users_list):
    # Your code using n_items and m_users
    # For example:
    print(f"n_items: {n_items}, m_users: {m_users}")
    # Run your analysis or processing using n_items and m_users here


n_items: 50, m_users: 50
n_items: 100, m_users: 100
n_items: 100, m_users: 300
n_items: 100, m_users: 5
n_items: 500, m_users: 100
n_items: 500, m_users: 500
n_items: 1000, m_users: 100
n_items: 1000, m_users: 200
n_items: 1000, m_users: 300
n_items: 1500, m_users: 1000
n_items: 1500, m_users: 2000


In [ ]:
#main loop 
import time as time_lib

k = 10 # k-fold validation 
n_items_list = [50, 100]#, 100, 100, 100]#, 500, 500, 500, 500, 500, 1000, 1000, 1000, 1000, 1000]
m_users_list = [ 50, 50]#, 300, 500, 700]#, 50, 100, 300, 500, 700,  50, 100, 300, 500, 700 ]
random_seed = 20230808

#prepare the result summary table 
result_summary_df = pd.DataFrame(columns = ['items','users','sparsity','avg_acc','std_acc','avg_rmse','std_rmse','avg_mad','std_mad','avg_time','std_time'])
# 6개의 열을 가진 빈 DataFrame 생성
result_detail_df = pd.DataFrame(columns=['n_items','m_users','acc', 'rmse', 'mad', 'time'])
#main 

for n, m in zip(n_items_list, m_users_list):
    n_items = n 
    n_users = m 

    ten = [str(j).zfill(2) for j in list(range(1,k+1))]
    
    labs = ['test'+i for i in ten]
    labs.insert(0,'orig')
    
    url_dict = {}
    url_dict["orig"] = './data/'+str(n_items)+'-by-'+str(n_users)+'_'+'original_matrix.csv'
    
    for i in ten:
        url_dict["test{0}".format(i)] = './data/'+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+i+'.csv'
    print(labs)
    
    for lab in labs:
        print(url_dict[lab])    
    
    df_dict = {}
    for lab in labs:
        #print(lab)
        df = pd.read_csv(url_dict[lab])
        #print(lab)
        df_dict[lab] = df.set_index('item_id')

    #compute the sparsity 
    n_row = df_dict["orig"].shape[0]
    n_col = df_dict["orig"].shape[1]
    Obs_ind = np.where(df_dict["orig"].notnull())    # Row and column indices for the observed entries of "Mdat"
    num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
    sparsity = 1 - num_Obs / (n_row * n_col)
    print("sparsity: "+str(sparsity))

    #Run algorithm 
    acc_list = []
    rmse_list = []
    mad_list = []
    time_list = []

    df_orig = df_dict["orig"]
    df_orig.columns = range(df_orig.shape[1])

    mm = df_orig.shape[0]
    nn = df_orig.shape[1]

    labs_test = labs[1:]

    col_max = [np.nanmax(df_dict["orig"].iloc[:, i].values) for i in range(df_dict["orig"].shape[1])]
    count = 0 

    for lab in labs_test:
        count += 1
        #print(lab)
        
        df = df_dict[lab]
        
        masked_df = pd.DataFrame(np.ma.masked_array(df, np.isnan(df)))
        #df, maxs = normal1(masked_df)
        #df, maxs = normal2(masked_df)
        #df, maxs, probs = normal3(masked_df)

        from missingpy import MissForest
        imputer = MissForest(verbose=0)

        start = time_lib.time()

        imputed = None 

        # Set a time limit of 3600 seconds (1 hour)
        time_limit = 30#3600
        
        try: 
            imputed = pd.DataFrame(imputer.fit_transform(df))
        except TimeoutError:
            print("at (n,m)= ("+str(n)+" ,"+str(m)+")")
            print("Imputation time exceeded the limit of {} seconds.".format(time_limit))
            imputed = None

        if imputed is not None:
            # Rounding for when no normalization           
            for i in range(mm):
                for j in range(nn):
                    x = imputed.iloc[i,j]
                    if x < 1:
                        imputed.iloc[i,j] = 1
                    elif x > col_max[j]:
                        imputed.iloc[i,j] = col_max[j]
                    else:
                        imputed.iloc[i,j] = round(x,0)

            #imputed = unnormal1(imputed, maxs)
            #imputed = unnormal2(imputed, maxs)
            #imputed = unnormal3(imputed, maxs, probs)

            end = time_lib.time()
            time = end - start

            df_orig.index = range(mm)
            imputed.index = range(mm)

            df_orig.columns = range(nn)
            imputed.columns = range(nn)

            #save the result data
            url = './result/'

            if count<10:
                filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data0'+str(count)+'_soft_imputed.csv'
            else:
                filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+str(count)+'_soft_imputed.csv'
            
            imputed.to_csv(filename)
            print("saved the result file as "+str(filename))
            diff = df_orig - imputed
            sq_diff = diff ** 2
            abs_diff = abs(diff)

            n_match = 0
            for i in range(mm):
                for j in range(nn):
                    if df_orig.isna().iloc[i,j]+df.isna().iloc[i,j]==1 and df_orig.iloc[i,j] == imputed.iloc[i,j]:
                        n_match += 1
            acc = n_match/(df.isna().sum().sum()-df_orig.isna().sum().sum())
            rmse = (sq_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())) ** 0.5
            mad = abs_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())
            
            acc_list.append(acc)
            rmse_list.append(rmse)
            mad_list.append(mad)
            time_list.append(time)
        else: #if time exceeds the limit 
            acc = 0 
            rmse = 0 
            mad = 0 
            time = time_limit    

            acc_list.append(acc)
            rmse_list.append(rmse)
            mad_list.append(mad)
            time_list.append(time)


    df = pd.DataFrame(data=[ acc_list, rmse_list,mad_list,time_list])
    display("n: "+str(n_items)+" ,m: "+ str(n_users))
    display(df.T)
    df_temp = df.T
    df_temp_2 = df_temp.copy()
    a = [n] * k 
    b = [m] * k
    df_temp_2.rename(columns={df_temp_2.columns[0]: 'acc', df_temp_2.columns[1]: 'rmse', df_temp_2.columns[2]: 'mad',df_temp_2.columns[3]: 'time'}, inplace=True)
    df_temp_2.insert(0,'m_users',b)
    df_temp_2.insert(0,'n_items',a) 

    # result_detail_df와 df_temp을 행 방향으로 이어 붙임
    result_detail_df = pd.concat([result_detail_df, df_temp_2], axis=0)

    filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data_all_missforest_imputed.csv'
    df_temp.to_csv(filename)
    print('saved '+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+ 'the summary result file as' +str(filename))
    # 열별 평균값 계산
    mean_values = df_temp.mean()
    # 열별 표준편차 계산
    std_values = df_temp.std()
    #new_row = [n,m,sparsity]+mean_values.tolist()
    new_row = [n,m,sparsity,mean_values[0],std_values[0],mean_values[1],std_values[1],mean_values[2],std_values[2],mean_values[3],std_values[3]]
    #display(new_row)
    result_summary_df = result_summary_df.append(pd.Series(new_row, index=result_summary_df.columns), ignore_index=True)
    filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'missforest_imputed.csv'
    df_temp.to_csv(filename)
    print("saved the summary result file as "+str(filename))
    display(result_summary_df)


display(result_summary_df)
display(result_detail_df)
        

        

['orig', 'test01', 'test02', 'test03', 'test04', 'test05', 'test06', 'test07', 'test08', 'test09', 'test10']
./data/50-by-50_original_matrix.csv
./data/20230808_50-by-50_10_fold_test_data01.csv
./data/20230808_50-by-50_10_fold_test_data02.csv
./data/20230808_50-by-50_10_fold_test_data03.csv
./data/20230808_50-by-50_10_fold_test_data04.csv
./data/20230808_50-by-50_10_fold_test_data05.csv
./data/20230808_50-by-50_10_fold_test_data06.csv
./data/20230808_50-by-50_10_fold_test_data07.csv
./data/20230808_50-by-50_10_fold_test_data08.csv
./data/20230808_50-by-50_10_fold_test_data09.csv
./data/20230808_50-by-50_10_fold_test_data10.csv
sparsity: 0.18400000000000005
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
saved the result file as ./result/20230808_50-by-50_10_fold_test_data01_soft_imputed.csv
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
saved the result file as ./result/20230808_50-by-50_10_fold_test_data02_soft_imputed.csv
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
I

'n: 50 ,m: 50'

,0,1,2,3
0,0.411765,1.033744,0.725490,19.511007
1,0.485294,0.868851,0.588235,19.441308
2,0.382353,0.954864,0.715686,24.433029
3,0.514706,0.851757,0.558824,31.081439
4,0.465686,0.975182,0.647059,31.647419
5,0.397059,0.982693,0.720588,25.946620
6,0.504902,0.910182,0.593137,20.442284
7,0.465686,0.947132,0.642157,20.050709
8,0.500000,0.928841,0.607843,25.168761
9,0.504902,0.931476,0.602941,35.294477


saved 20230808_50-by-50the summary result file as./result/20230808_50-by-50_10_fold_test_data_all_soft_imputed.csv
saved the summary result file as ./result/20230808_50-by-50_soft_imputed.csv


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,50.0,50.0,0.184,0.463235,0.048924,0.938472,0.053782,0.640196,0.060972,25.301705,5.713342


...

,0,1,2,3
0,0.476316,0.981406,0.657895,32.179989
1,0.513158,0.910465,0.592105,32.347623
2,0.460526,1.001315,0.681579,16.154387
3,0.468421,0.917663,0.626316,21.349230
4,0.494737,0.913351,0.607895,21.279999
5,0.486842,0.952835,0.634211,26.581575
6,0.465789,1.025978,0.673684,53.195878
7,0.443570,0.896186,0.635171,26.525019
8,0.419948,0.985459,0.692913,32.116624
9,0.451444,0.937691,0.648294,32.222791


saved 20230808_100-by-50the summary result file as./result/20230808_100-by-50_10_fold_test_data_all_soft_imputed.csv
saved the summary result file as ./result/20230808_100-by-50_soft_imputed.csv


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,50.0,50.0,0.1840,0.463235,0.048924,0.938472,0.053782,0.640196,0.060972,25.301705,5.713342
1,100.0,50.0,0.2394,0.468075,0.026707,0.952235,0.044230,0.645006,0.032267,29.395311,10.112771


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,50.0,50.0,0.1840,0.463235,0.048924,0.938472,0.053782,0.640196,0.060972,25.301705,5.713342
1,100.0,50.0,0.2394,0.468075,0.026707,0.952235,0.044230,0.645006,0.032267,29.395311,10.112771


,n_items,m_users,acc,rmse,mad,time
0,50,50,0.411765,1.033744,0.725490,19.511007
1,50,50,0.485294,0.868851,0.588235,19.441308
2,50,50,0.382353,0.954864,0.715686,24.433029
3,50,50,0.514706,0.851757,0.558824,31.081439
4,50,50,0.465686,0.975182,0.647059,31.647419
5,50,50,0.397059,0.982693,0.720588,25.946620
6,50,50,0.504902,0.910182,0.593137,20.442284
7,50,50,0.465686,0.947132,0.642157,20.050709
8,50,50,0.500000,0.928841,0.607843,25.168761
9,50,50,0.504902,0.931476,0.602941,35.294477


In [34]:
import time
import threading

# 실행 시간을 체크하고자 하는 함수
def long_running_function(exit_event):
    from missingpy import MissForest
    imputed = None
    imputer = MissForest(verbose=0)
    print("Long running function started.")
    start_time = time.time()
    #imputed = pd.DataFrame(imputer.fit_transform(df))
    while not exit_event.is_set() and time.time() - start_time < 100:
        time.sleep(100)
        imputed = 10 
    print("Long running function completed.")
    #function_thread.result = imputed 
    
# 함수 실행 상태 체크하는 스레드 함수
def check_function_status(target_thread,exit_event):
    start_time = time.time()
    while not exit_event.is_set():
        if time.time() - start_time > 15:
            print("Function execution time exceeded 15 seconds. Stopping...")
            exit_event.set()  # 스레드 중단
            target_thread.join()
            break
        print("Function is still running...")
        time.sleep(10) #매 10초 마다
    
    print("Function has completed.")

exit_event = threading.Event()
# 실행 시간을 체크하고자 하는 함수를 스레드에 전달하여 실행
function_thread = threading.Thread(target=long_running_function,args=(exit_event,))
#function_thread.result = None
function_thread.start()

# 함수 실행 상태 체크 스레드 실행
status_thread = threading.Thread(target=check_function_status, args=(function_thread,exit_event))
status_thread.start()

# 메인 스레드 대기
status_thread.join()
function_thread.join()

if function_thread.is_alive():
    print("Function execution was forcibly stopped.")
else:
    result = function_thread.result  # 함수 실행 결과값 가져오기
    #result = result_dict['result']
    print("Function result:", result)

print("Main thread completed.")


Long running function started.
Function is still running...
Function is still running...
Function execution time exceeded 15 seconds. Stopping...
Long running function completed.
Function has completed.


AttributeError: 'Thread' object has no attribute 'result'

In [12]:
#main loop 
import time as time_lib
import multiprocessing
from missingpy import MissForest

k = 10 # k-fold validation 
n_items_list = [50, 100]#, 100, 100, 100]#, 500, 500, 500, 500, 500, 1000, 1000, 1000, 1000, 1000]
m_users_list = [ 50, 50]#, 300, 500, 700]#, 50, 100, 300, 500, 700,  50, 100, 300, 500, 700 ]
random_seed = 20230808

#prepare the result summary table 
result_summary_df = pd.DataFrame(columns = ['items','users','sparsity','avg_acc','std_acc','avg_rmse','std_rmse','avg_mad','std_mad','avg_time','std_time'])
# 6개의 열을 가진 빈 DataFrame 생성
result_detail_df = pd.DataFrame(columns=['n_items','m_users','acc', 'rmse', 'mad', 'time'])
#main 

for n, m in zip(n_items_list, m_users_list):
    n_items = n 
    n_users = m 

    ten = [str(j).zfill(2) for j in list(range(1,k+1))]
    
    labs = ['test'+i for i in ten]
    labs.insert(0,'orig')
    
    url_dict = {}
    url_dict["orig"] = './data/'+str(n_items)+'-by-'+str(n_users)+'_'+'original_matrix.csv'
    
    for i in ten:
        url_dict["test{0}".format(i)] = './data/'+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+i+'.csv'
    print(labs)
    
    for lab in labs:
        print(url_dict[lab])    
    
    df_dict = {}
    for lab in labs:
        #print(lab)
        df = pd.read_csv(url_dict[lab])
        #print(lab)
        df_dict[lab] = df.set_index('item_id')

    #compute the sparsity 
    n_row = df_dict["orig"].shape[0]
    n_col = df_dict["orig"].shape[1]
    Obs_ind = np.where(df_dict["orig"].notnull())    # Row and column indices for the observed entries of "Mdat"
    num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
    sparsity = 1 - num_Obs / (n_row * n_col)
    print("sparsity: "+str(sparsity))

    #Run algorithm 
    acc_list = []
    rmse_list = []
    mad_list = []
    time_list = []

    df_orig = df_dict["orig"]
    df_orig.columns = range(df_orig.shape[1])

    mm = df_orig.shape[0]
    nn = df_orig.shape[1]

    labs_test = labs[1:]

    col_max = [np.nanmax(df_dict["orig"].iloc[:, i].values) for i in range(df_dict["orig"].shape[1])]
    count = 0 

    for lab in labs_test:
        count += 1
        #print(lab)
        
        df = df_dict[lab]
        
        masked_df = pd.DataFrame(np.ma.masked_array(df, np.isnan(df)))
        #df, maxs = normal1(masked_df)
        #df, maxs = normal2(masked_df)
        #df, maxs, probs = normal3(masked_df)

        from missingpy import MissForest
        imputer = MissForest(verbose=0)

        

        imputed = None 

        # Set a time limit of 3600 seconds (1 hour)
        time_limit = 30#3600

        while True: 
            start = time_lib.time()
            end = time_lib.time()
            time = end - start
            print(time)
            if time > time_limit: 
                acc = 0 
                rmse = 0 
                mad = 0 
                time = time_limit   
                acc_list.append(acc)
                rmse_list.append(rmse)
                mad_list.append(mad)
                time_list.append(time)
                print("Imputation time exceeded the limit of {} seconds.".format(time_limit))
                break
        
            imputed = pd.DataFrame(imputer.fit_transform(df))
        
            if imputed is not None:
                # Rounding for when no normalization           
                for i in range(mm):
                    for j in range(nn):
                        x = imputed.iloc[i,j]
                        if x < 1:
                            imputed.iloc[i,j] = 1
                        elif x > col_max[j]:
                            imputed.iloc[i,j] = col_max[j]
                        else:
                            imputed.iloc[i,j] = round(x,0)
                
                end = time_lib.time()
                time = end - start

                df_orig.index = range(mm)
                imputed.index = range(mm)

                df_orig.columns = range(nn)
                imputed.columns = range(nn)

                #save the result data
                url = './result/'

                if count<10:
                    filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data0'+str(count)+'_soft_imputed.csv'
                else:
                    filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+str(count)+'_soft_imputed.csv'
                
                imputed.to_csv(filename)
                print("saved the result file as "+str(filename))
                diff = df_orig - imputed
                sq_diff = diff ** 2
                abs_diff = abs(diff)

                n_match = 0
                for i in range(mm):
                    for j in range(nn):
                        if df_orig.isna().iloc[i,j]+df.isna().iloc[i,j]==1 and df_orig.iloc[i,j] == imputed.iloc[i,j]:
                            n_match += 1
                acc = n_match/(df.isna().sum().sum()-df_orig.isna().sum().sum())
                rmse = (sq_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())) ** 0.5
                mad = abs_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())
                
                acc_list.append(acc)
                rmse_list.append(rmse)
                mad_list.append(mad)
                time_list.append(time)

                break

            
    df = pd.DataFrame(data=[ acc_list, rmse_list,mad_list,time_list])
    display("n: "+str(n_items)+" ,m: "+ str(n_users))
    display(df.T)
    df_temp = df.T
    df_temp_2 = df_temp.copy()
    a = [n] * k 
    b = [m] * k
    df_temp_2.rename(columns={df_temp_2.columns[0]: 'acc', df_temp_2.columns[1]: 'rmse', df_temp_2.columns[2]: 'mad',df_temp_2.columns[3]: 'time'}, inplace=True)
    df_temp_2.insert(0,'m_users',b)
    df_temp_2.insert(0,'n_items',a) 

    # result_detail_df와 df_temp을 행 방향으로 이어 붙임
    result_detail_df = pd.concat([result_detail_df, df_temp_2], axis=0)

    filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data_all_missforest_imputed.csv'
    df_temp.to_csv(filename)
    print('saved '+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+ 'the summary result file as' +str(filename))
    # 열별 평균값 계산
    mean_values = df_temp.mean()
    # 열별 표준편차 계산
    std_values = df_temp.std()
    #new_row = [n,m,sparsity]+mean_values.tolist()
    new_row = [n,m,sparsity,mean_values[0],std_values[0],mean_values[1],std_values[1],mean_values[2],std_values[2],mean_values[3],std_values[3]]
    #display(new_row)
    result_summary_df = result_summary_df.append(pd.Series(new_row, index=result_summary_df.columns), ignore_index=True)
    filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'missforest_imputed.csv'
    df_temp.to_csv(filename)
    print("saved the summary result file as "+str(filename))
    display(result_summary_df)


display(result_summary_df)
display(result_detail_df)
        

        

['orig', 'test01', 'test02', 'test03', 'test04', 'test05', 'test06', 'test07', 'test08', 'test09', 'test10']
./data/50-by-50_original_matrix.csv
./data/20230808_50-by-50_10_fold_test_data01.csv
./data/20230808_50-by-50_10_fold_test_data02.csv
./data/20230808_50-by-50_10_fold_test_data03.csv
./data/20230808_50-by-50_10_fold_test_data04.csv
./data/20230808_50-by-50_10_fold_test_data05.csv
./data/20230808_50-by-50_10_fold_test_data06.csv
./data/20230808_50-by-50_10_fold_test_data07.csv
./data/20230808_50-by-50_10_fold_test_data08.csv
./data/20230808_50-by-50_10_fold_test_data09.csv
./data/20230808_50-by-50_10_fold_test_data10.csv
sparsity: 0.18400000000000005
0.0
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
saved the result file as ./result/20230808_50-by-50_10_fold_test_data01_soft_imputed.csv
0.0
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
Iteration: 5
saved the result file as ./result/20230808_50-by-50_10_fold_test_data02_soft_imputed.csv
0.0
It

'n: 50 ,m: 50'

,0,1,2,3
0,0.401961,1.004890,0.725490,24.045009
1,0.475490,0.910182,0.612745,29.090499
2,0.446078,0.967613,0.671569,50.345882
3,0.500000,0.860346,0.573529,25.360057
4,0.465686,0.947132,0.642157,46.345804
5,0.426471,0.977692,0.700980,36.071730
6,0.504902,0.902067,0.588235,21.014559
7,0.480392,0.928841,0.617647,19.665659
8,0.455882,0.957427,0.651961,35.624933
9,0.480392,0.995086,0.656863,30.431641


saved 20230808_50-by-50the summary result file as./result/20230808_50-by-50_10_fold_test_data_all_missforest_imputed.csv
saved the summary result file as ./result/20230808_50-by-50_missforest_imputed.csv


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,50.0,50.0,0.184,0.463725,0.032119,0.945128,0.045114,0.644118,0.047873,31.799577,10.34302


...

,0,1,2,3
0,0.423684,0.998683,0.702632,22.473263
1,0.515789,0.882580,0.573684,32.805850
2,0.471053,0.963819,0.650000,15.616591
3,0.447368,0.926226,0.647368,20.828895
4,0.494737,0.911910,0.610526,26.351050
5,0.444737,1.027260,0.692105,41.900626
6,0.497368,0.970621,0.631579,21.055577
7,0.464567,0.893253,0.619423,32.241581
8,0.425197,0.950204,0.671916,30.593821
9,0.482940,0.884394,0.598425,25.527981


saved 20230808_100-by-50the summary result file as./result/20230808_100-by-50_10_fold_test_data_all_missforest_imputed.csv
saved the summary result file as ./result/20230808_100-by-50_missforest_imputed.csv


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,50.0,50.0,0.1840,0.463725,0.032119,0.945128,0.045114,0.644118,0.047873,31.799577,10.343020
1,100.0,50.0,0.2394,0.466744,0.031363,0.940895,0.049679,0.639766,0.041230,26.939523,7.617011


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,50.0,50.0,0.1840,0.463725,0.032119,0.945128,0.045114,0.644118,0.047873,31.799577,10.343020
1,100.0,50.0,0.2394,0.466744,0.031363,0.940895,0.049679,0.639766,0.041230,26.939523,7.617011


,n_items,m_users,acc,rmse,mad,time
0,50,50,0.401961,1.004890,0.725490,24.045009
1,50,50,0.475490,0.910182,0.612745,29.090499
2,50,50,0.446078,0.967613,0.671569,50.345882
3,50,50,0.500000,0.860346,0.573529,25.360057
4,50,50,0.465686,0.947132,0.642157,46.345804
5,50,50,0.426471,0.977692,0.700980,36.071730
6,50,50,0.504902,0.902067,0.588235,21.014559
7,50,50,0.480392,0.928841,0.617647,19.665659
8,50,50,0.455882,0.957427,0.651961,35.624933
9,50,50,0.480392,0.995086,0.656863,30.431641
